In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
df_full = pd.read_csv("007_manchester_processed.csv")
print(df_full.isna().sum())

# 排序（时间序列必须）
df = df_full.sort_values(by='time')

df = df_full.drop(columns=['lat', 'lon', 'time', 'date'], errors='ignore')


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


time          0
TREFMXAV_U    0
FLNS          0
FSNS          0
PRECT         0
PRSN          0
QBOT          0
TREFHT        0
UBOT          0
VBOT          0
lat           0
lon           0
year          0
month         0
day           0
dayofyear     0
dtype: int64


In [2]:
#dataset split
train_df = df[(df['year'] >= 2006) & (df['year'] <= 2040)]
valid_df = df[(df['year'] > 2040) & (df['year'] < 2050)]
test_df  = df[(df['year'] >= 2050) & (df['year'] <= 2080)]

target = 'TREFMXAV_U'
features = [col for col in train_df.columns if col != target]

X_train, y_train = train_df[features], train_df[target]
X_valid, y_valid = valid_df[features], valid_df[target]
X_test,  y_test  = test_df[features],  test_df[target]

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# validation
y_valid_pred = rf.predict(X_valid)

val_mae = mean_absolute_error(y_valid, y_valid_pred)
val_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_pred))

print("===== Validation Results =====")
print("MAE:", val_mae)
print("RMSE:", val_rmse)

===== Validation Results =====
MAE: 0.5709293928490685
RMSE: 0.7562603956449583


In [4]:
from sklearn.metrics import r2_score, max_error

y_test_pred = rf.predict(X_test)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)
test_max_err = max_error(y_test, y_test_pred)

print("\n===== Test Results =====")
print("MAE:", test_mae)
print("RMSE:", test_rmse)
print("R²:", test_r2)
print("Max Error:", test_max_err)


===== Test Results =====
MAE: 0.5819213041010316
RMSE: 0.770180980798084
R²: 0.9798755052388806
Max Error: 5.082511033333411


In [5]:
from xgboost import XGBRegressor
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# training
xgb.fit(X_train, y_train)


y_valid_pred_xgb = xgb.predict(X_valid)

xgb_val_mae = mean_absolute_error(y_valid, y_valid_pred_xgb)
xgb_val_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_pred_xgb))

print("\n===== XGBoost Validation =====")
print("MAE:", xgb_val_mae)
print("RMSE:", xgb_val_rmse)




===== XGBoost Validation =====
MAE: 0.525295995690138
RMSE: 0.7034580030153887


In [6]:
# final training
X_final = pd.concat([X_train, X_valid])
y_final = pd.concat([y_train, y_valid])

xgb.fit(X_final, y_final)

# test
y_test_pred_xgb = xgb.predict(X_test)

xgb_test_mae = mean_absolute_error(y_test, y_test_pred_xgb)
xgb_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_xgb))
xgb_test_r2 = r2_score(y_test, y_test_pred_xgb)
xgb_test_max_err = max_error(y_test, y_test_pred_xgb)

print("\n===== XGBoost Test Results =====")
print("MAE:", xgb_test_mae)
print("RMSE:", xgb_test_rmse)
print("R²:", xgb_test_r2)
print("Max Error:", xgb_test_max_err)


===== XGBoost Test Results =====
MAE: 0.5275908424812704
RMSE: 0.7116840980225715
R²: 0.9828164088716351
Max Error: 4.96046570129397


In [7]:
from lightgbm import LGBMRegressor

# LGBM model
lgb = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# training
lgb.fit(X_train, y_train)

# Validation
y_valid_pred_lgb = lgb.predict(X_valid)

lgb_val_mae = mean_absolute_error(y_valid, y_valid_pred_lgb)
lgb_val_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_pred_lgb))

print("\n===== LightGBM Validation =====")
print("MAE:", lgb_val_mae)
print("RMSE:", lgb_val_rmse)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000086 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2581
[LightGBM] [Info] Number of data points in the train set: 12341, number of used features: 12
[LightGBM] [Info] Start training from score 14.646985

===== LightGBM Validation =====
MAE: 0.5231514822834773
RMSE: 0.7035909378190393


In [8]:
# ========= 最终训练（train + valid）=========
lgb.fit(X_final, y_final)

# ========= Test评估 =========
y_test_pred_lgb = lgb.predict(X_test)

lgb_test_mae = mean_absolute_error(y_test, y_test_pred_lgb)
lgb_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_lgb))
lgb_test_r2 = r2_score(y_test, y_test_pred_lgb)
lgb_test_max_err = max_error(y_test, y_test_pred_lgb)

print("\n===== LightGBM Test Results =====")
print("MAE:", lgb_test_mae)
print("RMSE:", lgb_test_rmse)
print("R²:", lgb_test_r2)
print("Max Error:", lgb_test_max_err)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000227 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2590
[LightGBM] [Info] Number of data points in the train set: 15487, number of used features: 12
[LightGBM] [Info] Start training from score 14.893295

===== LightGBM Test Results =====
MAE: 0.5227956567143205
RMSE: 0.700036511664648
R²: 0.9833742674304533
Max Error: 5.246589522913869
